# Companion notebook

Creates the deploy annotations, SLO, and three Agent Builder tools described in the article


In [ ]:
%pip install requests dotenv -q

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

KIBANA_URL = os.getenv("KIBANA_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
SERVICE_NAME = "opbeans-node"

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

print(f"Kibana URL: {KIBANA_URL}")
print(f"Service: {SERVICE_NAME}")

## Deploy annotations

In [ ]:
from datetime import datetime, timezone, timedelta

now = datetime.now(timezone.utc)

annotations = [
    {
        "@timestamp": (now - timedelta(hours=3)).isoformat().replace("+00:00", "Z"),
        "service": {"version": "2.0.9", "environment": "production"},
        "message": "Deployed v2.0.9 - fixed cart calculation bug",
    },
    {
        "@timestamp": (now - timedelta(hours=1)).isoformat().replace("+00:00", "Z"),
        "service": {"version": "2.1.0", "environment": "production"},
        "message": "Deployed v2.1.0 - updated product aggregation logic",
    },
]

for ann in annotations:
    response = requests.post(
        f"{KIBANA_URL}/api/apm/services/{SERVICE_NAME}/annotation",
        headers=headers,
        json=ann,
    )
    print(f"Annotation '{ann['service']['version']}': {response.status_code}")
    print(response.json())

## SLO

In [ ]:
slo_payload = {
    "name": "Service Availability",
    "description": f"{SERVICE_NAME} overall availability SLO",
    "indicator": {
        "type": "sli.apm.transactionErrorRate",
        "params": {
            "service": SERVICE_NAME,
            "environment": "production",
            "transactionType": "*",
            "transactionName": "*",
            "index": "metrics-apm.transaction.1m-*",
        },
    },
    "timeWindow": {"duration": "30d", "type": "rolling"},
    "objective": {"target": 0.995},
    "budgetingMethod": "occurrences",
}

response = requests.post(
    f"{KIBANA_URL}/api/observability/slos",
    headers=headers,
    json=slo_payload,
)
print(f"SLO created: {response.status_code}")
slo_data = response.json()
print(json.dumps(slo_data, indent=2))

SLO_ID = slo_data.get("id")
print(f"\nSLO_ID = {SLO_ID}")

## Tool 1: get_endpoint_health

In [ ]:
endpoint_health_tool = {
    "id": "get_endpoint_health",
    "type": "esql",
    "description": (
        "Returns the current health of a service endpoint over the last hour: error rate, "
        "latency percentiles (p50/p95/p99), and throughput per minute. Use this tool to assess "
        "whether a service is healthy before making changes. Interpretation guide: error rate "
        "below 0.5% is healthy, 0.5-1% is elevated (check for recent deploys), above 1% "
        "indicates a problem. For latency, compare p99 against the service baseline: checkout "
        "is typically under 500ms, product-search under 200ms. A sudden p99 spike within 15 "
        "minutes of a deploy suggests the deploy caused a regression."
    ),
    "tags": ["apm", "reliability", "health"],
    "configuration": {
        "query": (
            "FROM traces-apm-* "
            "| WHERE service.name == ?serviceName "
            "AND @timestamp >= NOW() - 1 hour "
            "AND transaction.duration.us IS NOT NULL "
            "| STATS total_transactions = COUNT(*), "
            'error_count = SUM(CASE(event.outcome == "failure", 1, 0)), '
            "p50_latency_ms = PERCENTILE(transaction.duration.us, 50) / 1000, "
            "p95_latency_ms = PERCENTILE(transaction.duration.us, 95) / 1000, "
            "p99_latency_ms = PERCENTILE(transaction.duration.us, 99) / 1000 "
            "BY service.name "
            "| EVAL error_rate_pct = ROUND(error_count * 100.0 / total_transactions, 2) "
            "| EVAL throughput_per_min = ROUND(total_transactions / 60.0, 1)"
        ),
        "params": {
            "serviceName": {
                "type": "keyword",
                "description": "The APM service name to check (e.g., opbeans-node)",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=endpoint_health_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

## Tool 2: get_recent_deploys

In [ ]:
recent_deploys_tool = {
    "id": "get_recent_deploys",
    "type": "esql",
    "description": (
        "Returns the deployment history for a service over the last 24 hours, including "
        "version numbers, timestamps, and deploy messages. Use this tool to understand the "
        "deployment timeline when assessing service health. Key patterns: if a deploy happened "
        "within the last 15 minutes, elevated error rates or latency may be expected (warm-up "
        "period). If metrics degraded immediately after a deploy, the deploy is the likely "
        "cause. Multiple deploys in a short window (under 2 hours) increase risk because it "
        "becomes harder to isolate which change caused an issue."
    ),
    "tags": ["apm", "deploys", "change-tracking"],
    "configuration": {
        "query": (
            "FROM observability-annotations "
            "| WHERE service.name == ?serviceName "
            "AND @timestamp >= NOW() - 24 hours "
            "| SORT @timestamp DESC "
            "| KEEP @timestamp, service.version, service.environment, message "
            "| LIMIT 10"
        ),
        "params": {
            "serviceName": {
                "type": "keyword",
                "description": "The APM service name to check deploy history for",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=recent_deploys_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

## Tool 3: get_slo_status

In [ ]:
slo_status_tool = {
    "id": "get_slo_status",
    "type": "esql",
    "description": (
        "Returns the current SLO burn rate for a service over the last hour. The response "
        "includes: SLI value (current performance), error budget target, and burn rate "
        "percentage. The burn rate tells you how fast the service is consuming error budget "
        "relative to the allowed threshold. Interpretation: a burn rate below 100% means the "
        "service is consuming budget slower than the limit, which is sustainable. Between "
        "100-200%, the service is burning budget faster than planned, so proceed with caution. "
        "Above 200%, the service is burning budget at double the allowed rate, so delay "
        "non-critical changes. Above 500%, investigate immediately. Note: this measures the "
        "*current* burn rate over the last hour, not cumulative budget consumption over the "
        "full SLO window. A temporarily high burn rate does not mean the overall budget is exhausted."
    ),
    "tags": ["slo", "reliability", "budget"],
    "configuration": {
        "query": (
            "FROM .slo-observability.sli-v* "
            "| WHERE slo.id == ?sloId "
            "AND @timestamp >= NOW() - 1 hour "
            "| STATS sli_value = AVG(slo.numerator) / AVG(slo.denominator) "
            "BY slo.id, slo.name "
            "| EVAL error_budget_target = 0.995 "
            "| EVAL burn_rate_pct = ROUND((1 - sli_value) / (1 - error_budget_target) * 100, 1)"
        ),
        "params": {
            "sloId": {
                "type": "keyword",
                "description": "The SLO identifier. Use the SLO ID for the service you are evaluating.",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=slo_status_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

## MCP client config

```json
{
  "mcpServers": {
    "elastic-agent-builder": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "https://YOUR_KIBANA_URL/api/agent_builder/mcp",
        "--header",
        "Authorization: ApiKey YOUR_BASE64_API_KEY"
      ]
    }
  }
}
```

## Cleanup

In [ ]:
for tool_id in ["get_endpoint_health", "get_recent_deploys", "get_slo_status"]:
    response = requests.delete(
        f"{KIBANA_URL}/api/agent_builder/tools/{tool_id}",
        headers=headers,
    )
    print(f"Deleted {tool_id}: {response.status_code}")

In [ ]:
if SLO_ID:
    response = requests.delete(
        f"{KIBANA_URL}/api/observability/slos/{SLO_ID}",
        headers=headers,
    )
    print(f"Deleted SLO {SLO_ID}: {response.status_code}")